In [30]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau # Import Learning Rate Scheduler
import warnings
warnings.filterwarnings('ignore') # Suppress warnings for cleaner output
import copy # To save the best model state

In [78]:


# --- Configuration ---
CUDA_DEVICE_ID = 0  # change if needed
DEVICE = (
    torch.device(f"cuda:{CUDA_DEVICE_ID}") if torch.cuda.is_available() else torch.device("cpu")
)
print(f"Using device: {DEVICE}")
BATCH_SIZE = 64
EPOCHS = 500 # Increased epochs significantly, relying on early stopping
LEARNING_RATE = 0.001
TEST_SPLIT_SIZE = 0.2 # Hold out 20% of training data for validation
RANDOM_SEED = 45
EARLY_STOPPING_PATIENCE = 40 # Number of epochs to wait for validation loss improvement
LR_SCHEDULER_PATIENCE = 30 # Number of epochs with no improvement after which learning rate will be reduced
LR_SCHEDULER_FACTOR = 0.5 # Factor by which the learning rate will be reduced

# --- Set Seed for Reproducibility ---
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# --- Load Data ---
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv or test.csv not found.")
    exit()

# --- Prepare Data ---
if 'target' in train_df.columns and all(col in train_df.columns for col in test_df.columns):
    X = train_df.drop('target', axis=1)
    y = train_df['target']
    X_test_final = test_df

    if list(X.columns) == list(X_test_final.columns):
        print("Train and test columns match.")

        # --- Preprocessing (Scaling) ---
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_test_scaled = scaler.transform(X_test_final)

        # --- Train/Validation Split ---
        X_train, X_val, y_train, y_val = train_test_split(
            X_scaled, y.values, test_size=TEST_SPLIT_SIZE, random_state=RANDOM_SEED, stratify=y
        )

        print(f"Data scaled and split.")
        print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test_scaled.shape}")

        # --- PyTorch Dataset ---
        class DroneDataset(Dataset):
            def __init__(self, features, labels=None):
                # Convert to float32, the default float type for PyTorch
                self.features = torch.tensor(features, dtype=torch.float32)
                # Ensure labels are also float32 for BCELoss and reshape to [n_samples, 1]
                self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) if labels is not None else None

            def __len__(self):
                return len(self.features)

            def __getitem__(self, idx):
                if self.labels is not None:
                    return self.features[idx], self.labels[idx]
                else:
                    # Return only features if labels are not provided (for test set)
                    return self.features[idx]

        # Create Datasets
        train_dataset = DroneDataset(X_train, y_train)
        val_dataset = DroneDataset(X_val, y_val)
        test_dataset = DroneDataset(X_test_scaled) # No labels for the final test set

        # Create DataLoaders
        train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False) # No need to shuffle test data

Using device: cuda:0
Data loaded successfully.
Train and test columns match.
Data scaled and split.
Train shape: (1030, 16), Validation shape: (258, 16), Test shape: (430, 16)


In [7]:
display(train_df.shape)
display(train_df)

(1288, 17)

,area,perimeter,major_axis,minor_axis,eccentricity,eqdiasq,solidity,convex_area,extent,aspect_ratio,roundness,compactness,shapefactor_1,shapefactor_2,shapefactor_3,shapefactor_4,target
0,75516,1731.4840,411.7352,245.7620,0.8023,310.0806,0.9148,82546,0.7169,1.6753,0.3165,0.7531,0.0055,0.0033,0.5672,0.9502,1
1,98903,1374.4370,477.2451,269.7676,0.8249,354.8622,0.9585,103181,0.7679,1.7691,0.6579,0.7436,0.0048,0.0027,0.5529,0.9781,0
2,84746,1311.1570,482.7735,235.9040,0.8725,328.4843,0.9121,92914,0.7162,2.0465,0.6195,0.6804,0.0057,0.0028,0.4630,0.9474,1
3,98184,1463.1680,434.3769,292.6472,0.7390,353.5700,0.9543,102890,0.7316,1.4843,0.5763,0.8140,0.0044,0.0030,0.6625,0.9834,0
4,94170,1267.7271,440.1109,278.4162,0.7745,346.2672,0.9643,97656,0.6836,1.5808,0.7363,0.7868,0.0047,0.0030,0.6190,0.9785,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1283,103151,2145.1340,499.3782,280.9169,0.8268,362.4030,0.9193,112202,0.7205,1.7777,0.2817,0.7257,0.0048,0.0027,0.5267,0.9362,0
1284,54322,1569.1990,363.6869,217.1759,0.8021,262.9922,0.8557,63481,0.6645,1.6746,0.2772,0.7231,0.0067,0.0040,0.5229,0.8757,1
1285,69080,1713.4390,434.3565,213.7013,0.8706,296.5727,0.9217,74947,0.6828,2.0325,0.2957,0.6828,0.0063,0.0031,0.4662,0.9476,1
1286,83490,1161.5470,448.3226,251.6644,0.8276,326.0410,0.9437,88472,0.6826,1.7814,0.7776,0.7272,0.0054,0.0030,0.5289,0.9422,0


# решение Вани

In [94]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau # Import Learning Rate Scheduler
import warnings
warnings.filterwarnings('ignore') # Suppress warnings for cleaner output
import copy # To save the best model state

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
BATCH_SIZE = 64
EPOCHS = 500 # Increased epochs significantly, relying on early stopping
LEARNING_RATE = 0.001
TEST_SPLIT_SIZE = 0.2 # Hold out 20% of training data for validation
RANDOM_SEED = 42
EARLY_STOPPING_PATIENCE = 40 # Number of epochs to wait for validation loss improvement
LR_SCHEDULER_PATIENCE = 30 # Number of epochs with no improvement after which learning rate will be reduced
LR_SCHEDULER_FACTOR = 0.5 # Factor by which the learning rate will be reduced

# --- Set Seed for Reproducibility ---
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# --- Load Data ---
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv or test.csv not found.")
    exit()

# --- Prepare Data ---
if 'target' in train_df.columns and all(col in train_df.columns for col in test_df.columns):
    X = train_df.drop('target', axis=1)
    y = train_df['target']
    X_test_final = test_df

    if list(X.columns) == list(X_test_final.columns):
        print("Train and test columns match.")

        # --- Preprocessing (Scaling) ---
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_test_scaled = scaler.transform(X_test_final)

        # --- Train/Validation Split ---
        X_train, X_val, y_train, y_val = train_test_split(
            X_scaled, y.values, test_size=TEST_SPLIT_SIZE, random_state=RANDOM_SEED, stratify=y
        )

        print(f"Data scaled and split.")
        print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test_scaled.shape}")

        # --- PyTorch Dataset ---
        class DroneDataset(Dataset):
            def __init__(self, features, labels=None):
                # Convert to float32, the default float type for PyTorch
                self.features = torch.tensor(features, dtype=torch.float32)
                # Ensure labels are also float32 for BCELoss and reshape to [n_samples, 1]
                self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) if labels is not None else None

            def __len__(self):
                return len(self.features)

            def __getitem__(self, idx):
                if self.labels is not None:
                    return self.features[idx], self.labels[idx]
                else:
                    # Return only features if labels are not provided (for test set)
                    return self.features[idx]

        # Create Datasets
        train_dataset = DroneDataset(X_train, y_train)
        val_dataset = DroneDataset(X_val, y_val)
        test_dataset = DroneDataset(X_test_scaled) # No labels for the final test set

        # Create DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False) # No need to shuffle test data


        # --- Neural Network Model ---
        class SimpleNN(nn.Module):
            def __init__(self, input_dim):
                super(SimpleNN, self).__init__()
                # Slightly increased neurons and added one more layer
                self.layer_1 = nn.Linear(input_dim, 256) # First hidden layer
                self.relu_1 = nn.ReLU()
                self.dropout_1 = nn.Dropout(0.4) # Slightly increased dropout
                self.layer_2 = nn.Linear(256, 128)   # Second hidden layer
                self.relu_2 = nn.ReLU()
                self.dropout_2 = nn.Dropout(0.4) # Slightly increased dropout
                self.layer_3 = nn.Linear(128, 64)   # Third hidden layer
                self.relu_3 = nn.ReLU()
                self.dropout_3 = nn.Dropout(0.3) # Dropout for the new layer
                self.output_layer = nn.Linear(64, 1) # Output layer
                self.sigmoid = nn.Sigmoid()         # Sigmoid activation

            def forward(self, x):
                x = self.layer_1(x)
                x = self.relu_1(x)
                x = self.dropout_1(x)
                x = self.layer_2(x)
                x = self.relu_2(x)
                x = self.dropout_2(x)
                x = self.layer_3(x)
                x = self.relu_3(x)
                x = self.dropout_3(x)
                x = self.output_layer(x)
                x = self.sigmoid(x)
                return x

        # Instantiate the model
        input_dimension = X_train.shape[1] # Number of features
        model = SimpleNN(input_dimension).to(DEVICE)
        print("Model Architecture:")
        print(model)

        # --- Loss Function and Optimizer ---
        criterion = nn.BCELoss() # Binary Cross Entropy Loss
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
        # Learning Rate Scheduler
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=LR_SCHEDULER_FACTOR, patience=LR_SCHEDULER_PATIENCE, verbose=True)


        # --- Training Loop ---
        best_val_loss = float('inf') # Initialize with infinity for early stopping
        epochs_no_improve = 0 # Counter for early stopping
        best_model_state = None # To store the state of the best model

        print("Starting Training...")
        for epoch in range(EPOCHS):
            model.train() # Set model to training mode
            train_loss = 0.0
            correct_train = 0
            total_train = 0

            for features, labels in train_loader:
                features, labels = features.to(DEVICE), labels.to(DEVICE)

                # Zero gradients
                optimizer.zero_grad()

                # Forward pass
                outputs = model(features)
                loss = criterion(outputs, labels)

                # Backward pass and optimize
                loss.backward()
                optimizer.step()

                # Track loss and accuracy
                train_loss += loss.item() * features.size(0)
                predicted = (outputs > 0.5).float() # Convert probabilities to 0/1
                total_train += labels.size(0)
                correct_train += (predicted == labels).sum().item()

            epoch_train_loss = train_loss / len(train_loader.dataset)
            epoch_train_acc = correct_train / total_train

            # --- Validation Step ---
            model.eval() # Set model to evaluation mode
            val_loss = 0.0
            correct_val = 0
            total_val = 0
            with torch.no_grad(): # Disable gradient calculation for validation
                for features, labels in val_loader:
                    features, labels = features.to(DEVICE), labels.to(DEVICE)
                    outputs = model(features)
                    loss = criterion(outputs, labels)

                    val_loss += loss.item() * features.size(0)
                    predicted = (outputs > 0.5).float()
                    total_val += labels.size(0)
                    correct_val += (predicted == labels).sum().item()


            epoch_val_loss = val_loss / len(val_loader.dataset)
            epoch_val_acc = correct_val / total_val

            print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

            # --- Learning Rate Scheduler Step ---
            scheduler.step(epoch_val_loss)

            # --- Early Stopping Check ---
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                epochs_no_improve = 0
                # Save the best model state
                best_model_state = copy.deepcopy(model.state_dict())
                # print(f"--- Best validation loss improved to {best_val_loss:.4f} ---")
            else:
                epochs_no_improve += 1
                if epochs_no_improve == EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping triggered after {EARLY_STOPPING_PATIENCE} epochs with no improvement in validation loss.")
                    break # Stop training loop

        print("Training Finished.")

        # Load the best model state before prediction
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            print("Loaded best model state based on validation loss.")
        else:
            print("No improvement in validation loss, using the last model state.")


        # --- Prediction on Final Test Set ---
        model.eval() # Ensure model is in evaluation mode
        all_predictions = []
        with torch.no_grad():
            for features in test_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                predicted_probs = outputs.cpu().numpy() # Move to CPU and convert to numpy
                predicted_classes = (predicted_probs > 0.5).astype(int) # Threshold at 0.5
                all_predictions.extend(predicted_classes.flatten()) # Flatten in case of single-column output

        print("Making predictions with the Neural Network model...")

        # --- Output Generation ---
        output_df_nn = pd.DataFrame(all_predictions, columns=None)
        output_filename_nn = 'answers.csv' # Changed filename to 'answers.csv' as required
        try:
            output_df_nn.to_csv(output_filename_nn, index=False, header=False)
            print(f"Neural Network predictions saved to {output_filename_nn}")

            # --- Feedback based on best validation loss (proxy for generalization) ---
            # Note: We don't have the true test accuracy here, but validation loss is a good indicator
            print("Validation metrics from the epoch with the best validation loss:")
            # You would need to track best validation accuracy alongside best validation loss
            # For simplicity here, we just report the best loss and remind about the target
            print(f"Best Validation Loss achieved: {best_val_loss:.4f}")
            print("Goal is to achieve accuracy between 86.5% and 91.5% on the unseen test set.")
            print(f"Your previous score suggests you are very close to the target range.")
            print(f"Try submitting the '{output_filename_nn}' file generated by this code.")


        except Exception as e:
            print(f"Error saving Neural Network predictions to {output_filename_nn}: {e}")

    else:
        print("Error: Training and test feature columns do not match.")
else:
    print("Error: 'target' column not found in train.csv or columns mismatch.")

Using device: cuda
Data loaded successfully.
Train and test columns match.
Data scaled and split.
Train shape: (1030, 16), Validation shape: (258, 16), Test shape: (430, 16)
Model Architecture:
SimpleNN(
  (layer_1): Linear(in_features=16, out_features=256, bias=True)
  (relu_1): ReLU()
  (dropout_1): Dropout(p=0.4, inplace=False)
  (layer_2): Linear(in_features=256, out_features=128, bias=True)
  (relu_2): ReLU()
  (dropout_2): Dropout(p=0.4, inplace=False)
  (layer_3): Linear(in_features=128, out_features=64, bias=True)
  (relu_3): ReLU()
  (dropout_3): Dropout(p=0.3, inplace=False)
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Starting Training...
Epoch 1/500 | Train Loss: 0.5806 | Train Acc: 0.7835 | Val Loss: 0.3965 | Val Acc: 0.8411
Epoch 2/500 | Train Loss: 0.3691 | Train Acc: 0.8505 | Val Loss: 0.3512 | Val Acc: 0.8527
Epoch 3/500 | Train Loss: 0.3416 | Train Acc: 0.8631 | Val Loss: 0.3330 | Val Acc: 0.8527
Epoch 4/500 | Train Loss

# простые методы

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.ensemble import GradientBoostingClassifier as GBC
from sklearn.ensemble import ExtraTreesClassifier as ETC
from sklearn.ensemble import AdaBoostClassifier as AdaBoost
from tqdm import tqdm

# Load data
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
except FileNotFoundError:
    print("train.csv or test.csv not found")
    exit()

# Prepare data
if 'target' not in train_df.columns:
    print("Target column not found in train.csv")
    exit()

X = train_df.drop('target', axis=1)
y = train_df['target']
X_test_final = test_df

# Ensure train and test columns match
if list(X.columns) != list(X_test_final.columns):
    print("Train and test columns do not match")
    exit()

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test_final = scaler.transform(X_test_final)

# Define parameter grids
param_grid_rfc = {'n_estimators': [100, 200, 300], 'max_depth': [5, 7, 9, 12], 'min_samples_split': [2, 4, 6]}
param_grid_gbc = {'n_estimators': [100, 200], 'learning_rate': [0.001,0.01, 0.1], 'max_depth': [3, 5, 7]}
param_grid_etc = {'n_estimators': [100, 200], 'max_depth': [5, 7], 'min_samples_split': [2, 4]}
param_grid_ada = {'n_estimators': [50, 100], 'learning_rate': [0.001,0.01, 0.1]}


# Models with GridSearchCV
rfc = RFC(random_state=42)
grid_search_rfc = GridSearchCV(rfc, param_grid_rfc, cv=3, scoring='accuracy', verbose=0)

gbc = GBC(random_state=42)
grid_search_gbc = GridSearchCV(gbc, param_grid_gbc, cv=3, scoring='accuracy', verbose=0)

etc = ETC(random_state=42)
grid_search_etc = GridSearchCV(etc, param_grid_etc, cv=3, scoring='accuracy', verbose=0)

ada = AdaBoost(random_state=42)
grid_search_ada = GridSearchCV(ada, param_grid_ada, cv=3, scoring='accuracy', verbose=0)


# Train models with tqdm
print("Training RFC...")
grid_search_rfc.fit(X_train, y_train)

print("Training GBC...")
grid_search_gbc.fit(X_train, y_train)

print("Training ETC...")
grid_search_etc.fit(X_train, y_train)

print("Training AdaBoost...")
grid_search_ada.fit(X_train, y_train)


# Best models
best_rfc = grid_search_rfc.best_estimator_
best_gbc = grid_search_gbc.best_estimator_
best_etc = grid_search_etc.best_estimator_
best_ada = grid_search_ada.best_estimator_

# Evaluate on validation set
rfc_pred_val = best_rfc.predict(X_val)
gbc_pred_val = best_gbc.predict(X_val)
etc_pred_val = best_etc.predict(X_val)
ada_pred_val = best_ada.predict(X_val)

print("RFC (Validation):")
print("Accuracy:", accuracy_score(y_val, rfc_pred_val))
print(classification_report(y_val, rfc_pred_val))

print("\nGBC (Validation):")
print("Accuracy:", accuracy_score(y_val, gbc_pred_val))
print(classification_report(y_val, gbc_pred_val))

print("\nETC (Validation):")
print("Accuracy:", accuracy_score(y_val, etc_pred_val))
print(classification_report(y_val, etc_pred_val))

print("\nAdaBoost (Validation):")
print("Accuracy:", accuracy_score(y_val, ada_pred_val))
print(classification_report(y_val, ada_pred_val))


# Make predictions on the final test set
rfc_pred_test = best_rfc.predict(X_test_final)
gbc_pred_test = best_gbc.predict(X_test_final)
etc_pred_test = best_etc.predict(X_test_final)
ada_pred_test = best_ada.predict(X_test_final)

print("\nPredictions on test.csv completed. (Predictions are stored in ..._pred_test)")


# Model Recommendation (Placeholder - Replace with analysis)
best_model = None
best_accuracy = 0

if accuracy_score(y_val, rfc_pred_val) > best_accuracy:
    best_accuracy = accuracy_score(y_val, rfc_pred_val)
    best_model = "Random Forest"

if accuracy_score(y_val, gbc_pred_val) > best_accuracy:
    best_accuracy = accuracy_score(y_val, gbc_pred_val)
    best_model = "Gradient Boosting"

if accuracy_score(y_val, etc_pred_val) > best_accuracy:
    best_accuracy = accuracy_score(y_val, etc_pred_val)
    best_model = "Extra Trees"

if accuracy_score(y_val, ada_pred_val) > best_accuracy:
    best_accuracy = accuracy_score(y_val, ada_pred_val)
    best_model = "AdaBoost"


print(f"\nBest model based on validation accuracy: {best_model} with accuracy {best_accuracy}")


print("\nRecommendation for Neural Network (NN):")
if best_model == "Random Forest":
    print("Since Random Forest performs well, consider a NN with moderate complexity. Experiment with different architectures.")
elif best_model == "Gradient Boosting":
    print("Gradient Boosting also indicates complex relationships. Try a slightly deeper NN or feature engineering before the NN.")
elif best_model == "Extra Trees":
    print("Extra Trees suggests feature importance. Consider feature selection or a NN that highlights important features.")
elif best_model == "AdaBoost":
    print("AdaBoost may indicate simpler patterns. A shallow NN might be sufficient.")
else:
    print("No clear best model. Experiment with various NN architectures and hyperparameters.")

# нейронкой

In [32]:
# Creating model instance
model_task_1 = nn.Sequential(

    nn.Linear(16, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(128, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(128, 1),
    nn.ReLU()
)
'''
nn.Conv1d(1, 64, 3, padding=1), # на входе 28x28
nn.BatchNorm2d(num_features=64),
nn.ReLU(),
nn.Conv1d(64, 32, 3, padding=1), # на входе 28x28
nn.BatchNorm2d(num_features=32),
nn.Conv1d(32, 16, 3, padding=1), # на входе 28x28
nn.BatchNorm2d(num_features=16),
nn.ReLU(),
nn.Flatten(),
#nn.Unflatten(1,(1,28,28)),
#nn.Linear(1*28*28, 128),
nn.Linear(16, 128),
#nn.BatchNorm1d(128),
nn.Linear(128, 10)
'''
model_task_1.to(DEVICE)
# your code here
# your code here
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_task_1.parameters(),lr=0.001)

n_ep = 5
for epoch in range(n_ep):
    model_task_1.train()  # Режим обучения
    running_loss = 0.0

    for batch_idx, (inputs, labels) in enumerate(train_data_loader):
        inputs, labels = inputs, labels
        #print(type(inputs))
        #inputs = torch.stack([m.reshape(-1,784) for m in inputs])
        #inputs = inputs.reshape(-1,784)
        # Обнуляем градиенты
        inputs = inputs.to('cuda')  # or inputs = inputs.cuda()
        labels = labels.to('cuda')
        optimizer.zero_grad()

        # Forward pass
        outputs = model_task_1(inputs)
        loss = criterion(outputs, labels)

        # Backward pass и оптимизация
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Логирование каждые 100 батчей
        print(f'Epoch {epoch+1}, Batch {batch_idx+1}, Loss: {running_loss / 100:.3f}')

    running_loss = 0.0
    print(f'Epoch {epoch+1} finished!')

Epoch 1, Batch 1, Loss: 0.000
Epoch 1, Batch 2, Loss: 0.000
Epoch 1, Batch 3, Loss: 0.000
Epoch 1, Batch 4, Loss: 0.000
Epoch 1, Batch 5, Loss: 0.000
Epoch 1, Batch 6, Loss: 0.000
Epoch 1, Batch 7, Loss: 0.000
Epoch 1, Batch 8, Loss: 0.000
Epoch 1, Batch 9, Loss: 0.000
Epoch 1, Batch 10, Loss: 0.000
Epoch 1, Batch 11, Loss: 0.000
Epoch 1, Batch 12, Loss: 0.000
Epoch 1, Batch 13, Loss: 0.000
Epoch 1, Batch 14, Loss: 0.000
Epoch 1, Batch 15, Loss: 0.000
Epoch 1, Batch 16, Loss: 0.000
Epoch 1, Batch 17, Loss: 0.000
Epoch 1 finished!
Epoch 2, Batch 1, Loss: 0.000
Epoch 2, Batch 2, Loss: 0.000
Epoch 2, Batch 3, Loss: 0.000
Epoch 2, Batch 4, Loss: 0.000
Epoch 2, Batch 5, Loss: 0.000
Epoch 2, Batch 6, Loss: 0.000
Epoch 2, Batch 7, Loss: 0.000
Epoch 2, Batch 8, Loss: 0.000
Epoch 2, Batch 9, Loss: 0.000
Epoch 2, Batch 10, Loss: 0.000
Epoch 2, Batch 11, Loss: 0.000
Epoch 2, Batch 12, Loss: 0.000
Epoch 2, Batch 13, Loss: 0.000
Epoch 2, Batch 14, Loss: 0.000
Epoch 2, Batch 15, Loss: 0.000
Epoch 2,

In [35]:
# Creating model instance
model_task_1 = nn.Sequential(
    nn.Linear(16, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(64, 1),
    nn.Sigmoid()  # Sigmoid for binary classification
)
model_task_1.to(DEVICE)
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model_task_1.parameters(), lr=LEARNING_RATE)
n_ep = EPOCHS
for epoch in range(n_ep):
    model_task_1.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(train_data_loader):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model_task_1(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        print(f'Epoch {epoch+1}, Batch {batch_idx+1}/{17}, Loss: {loss.item():.3f}')
    print(f'Epoch {epoch+1} finished!')


Выходные данные были обрезаны до нескольких последних строк (5000).
Epoch 223, Batch 8/17, Loss: 0.303
Epoch 223, Batch 9/17, Loss: 0.269
Epoch 223, Batch 10/17, Loss: 0.258
Epoch 223, Batch 11/17, Loss: 0.216
Epoch 223, Batch 12/17, Loss: 0.316
Epoch 223, Batch 13/17, Loss: 0.229
Epoch 223, Batch 14/17, Loss: 0.258
Epoch 223, Batch 15/17, Loss: 0.314
Epoch 223, Batch 16/17, Loss: 0.292
Epoch 223, Batch 17/17, Loss: 0.627
Epoch 223 finished!
Epoch 224, Batch 1/17, Loss: 0.220
Epoch 224, Batch 2/17, Loss: 0.227
Epoch 224, Batch 3/17, Loss: 0.260
Epoch 224, Batch 4/17, Loss: 0.267
Epoch 224, Batch 5/17, Loss: 0.231
Epoch 224, Batch 6/17, Loss: 0.266
Epoch 224, Batch 7/17, Loss: 0.290
Epoch 224, Batch 8/17, Loss: 0.244
Epoch 224, Batch 9/17, Loss: 0.267
Epoch 224, Batch 10/17, Loss: 0.305
Epoch 224, Batch 11/17, Loss: 0.285
Epoch 224, Batch 12/17, Loss: 0.191
Epoch 224, Batch 13/17, Loss: 0.370
Epoch 224, Batch 14/17, Loss: 0.263
Epoch 224, Batch 15/17, Loss: 0.220
Epoch 224, Batch 16/17,

In [37]:
def get_accuracy(model, data_loader):
    predicted_labels = []
    real_labels = []
    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            y_predicted = model(batch[0].to(DEVICE))
            predicted_labels.append(y_predicted.argmax(dim=1).cpu())
            real_labels.append(batch[1])

    predicted_labels = torch.cat(predicted_labels)
    real_labels = torch.cat(real_labels)
    accuracy_score = (predicted_labels == real_labels).type(torch.FloatTensor).mean()
    return accuracy_score
test_acc_task = get_accuracy(model_task_1, test_data_loader)
print(f"Neural network accuracy on test set: {test_acc_task:3.5}")

ValueError: expected 2D or 3D input (got 1D input)

In [39]:
def get_accuracy(model, data_loader):
    predicted_labels = []
    real_labels = []
    model.eval()
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE) # Move labels to device as well
            outputs = model(inputs).squeeze() # Remove the extra dimension
            predicted = (outputs > 0.5).float() # Convert probabilities to binary predictions
            predicted_labels.extend(predicted.cpu().numpy())
            real_labels.extend(labels.cpu().numpy().flatten())

    # Convert lists to numpy arrays
    predicted_labels = np.array(predicted_labels)
    real_labels = np.array(real_labels)

    accuracy = accuracy_score(real_labels, predicted_labels)
    return accuracy

test_acc_task = get_accuracy(model_task_1, test_data_loader)
print(f"Neural network accuracy on test set: {test_acc_task:3.5f}")

ValueError: too many values to unpack (expected 2)

In [ ]:
# --- Test on test.csv ---
model_task_1.eval()  # Set the model to evaluation mode
test_predictions = []
with torch.no_grad():  # Disable gradient calculation during inference
    for inputs in test_data_loader:
        inputs = inputs.to(DEVICE)
        outputs = model_task_1(inputs)
        # Assuming outputs are probabilities, threshold at 0.5 to get class labels
        predicted = (outputs > 0.5).float()
        test_predictions.extend(predicted.cpu().numpy())
# Convert predictions to 0/1
test_predictions = [int(x) for x in test_predictions]
print("\nPredictions on test.csv completed.  First 20 predictions:")
print(test_predictions[:20])

In [27]:
len(train_data_loader)

17

#ver2

In [46]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 0.001
EPOCHS = 10
BATCH_SIZE = 32

# Define Dataset
class MyDataset(Dataset):
    def __init__(self, data_file, transform=None):
        df = pd.read_csv(data_file)
        self.inputs = df.iloc[:, :-1].values  # All columns except the last one
        self.labels = df.iloc[:, -1].values  # Last column as labels
        self.transform = transform

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        inp = self.inputs[idx].astype(np.float32)
        lab = self.labels[idx].astype(np.float32)
        if self.transform:
            inp = self.transform(inp)
        return torch.tensor(inp), torch.tensor(lab).unsqueeze(0)


# Load Data
train_dataset = MyDataset('train.csv')
test_dataset = MyDataset('test.csv')

train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# Model
model = nn.Sequential(
    nn.Linear(16, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(64, 1),
    nn.Sigmoid()
).to(DEVICE)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# Train & Evaluate
def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, epochs, device):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            print(f'Epoch {epoch+1}, Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.3f}')
        print(f'Epoch {epoch+1} finished!')
# Run
train_and_evaluate(model, train_dl, test_dl, criterion, optimizer, EPOCHS, DEVICE)

Epoch 1, Batch 1/41, Loss: 0.744
Epoch 1, Batch 2/41, Loss: 0.830
Epoch 1, Batch 3/41, Loss: 0.705
Epoch 1, Batch 4/41, Loss: 0.778
Epoch 1, Batch 5/41, Loss: 0.598
Epoch 1, Batch 6/41, Loss: 0.609
Epoch 1, Batch 7/41, Loss: 0.655
Epoch 1, Batch 8/41, Loss: 0.617
Epoch 1, Batch 9/41, Loss: 0.526
Epoch 1, Batch 10/41, Loss: 0.613
Epoch 1, Batch 11/41, Loss: 0.528
Epoch 1, Batch 12/41, Loss: 0.557
Epoch 1, Batch 13/41, Loss: 0.612
Epoch 1, Batch 14/41, Loss: 0.481
Epoch 1, Batch 15/41, Loss: 0.506
Epoch 1, Batch 16/41, Loss: 0.575
Epoch 1, Batch 17/41, Loss: 0.470
Epoch 1, Batch 18/41, Loss: 0.504
Epoch 1, Batch 19/41, Loss: 0.594
Epoch 1, Batch 20/41, Loss: 0.469
Epoch 1, Batch 21/41, Loss: 0.561
Epoch 1, Batch 22/41, Loss: 0.495
Epoch 1, Batch 23/41, Loss: 0.337
Epoch 1, Batch 24/41, Loss: 0.506
Epoch 1, Batch 25/41, Loss: 0.476
Epoch 1, Batch 26/41, Loss: 0.634
Epoch 1, Batch 27/41, Loss: 0.419
Epoch 1, Batch 28/41, Loss: 0.450
Epoch 1, Batch 29/41, Loss: 0.403
Epoch 1, Batch 30/41, L

In [48]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 0.001
EPOCHS = 10
BATCH_SIZE = 64

# Define Dataset
class MyDataset(Dataset):
    def __init__(self, data_file, is_train=True, transform=None):
        df = pd.read_csv(data_file)
        self.is_train = is_train
        self.inputs = df.iloc[:, :-1].values if is_train else df.values
        self.labels = df.iloc[:, -1].values if is_train else None
        self.transform = transform

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        inp = self.inputs[idx].astype(np.float32)
        if self.is_train:
            lab = self.labels[idx].astype(np.float32)
            return torch.tensor(inp), torch.tensor(lab).unsqueeze(0)
        else:
            return torch.tensor(inp)


# Load Data
train_dataset = MyDataset('train.csv')
test_dataset = MyDataset('test.csv', is_train=False)

train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# Model
model = nn.Sequential(
    nn.Linear(16, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(p=0.4),
    nn.Linear(64, 1),
    nn.Sigmoid()
).to(DEVICE)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


# Train & Evaluate
def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, epochs, device):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            print(f'Epoch {epoch+1}, Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.3f}')
        print(f'Epoch {epoch+1} finished!')

    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = (outputs > 0.5).int().flatten().cpu().numpy()
            predictions.extend(preds)

    return predictions

# Run
predictions = train_and_evaluate(model, train_dl, test_dl, criterion, optimizer, EPOCHS, DEVICE)

df_answers = pd.DataFrame(predictions, columns=['predictions'])
df_answers.to_csv('answers.csv', index=False)


Epoch 1, Batch 1/21, Loss: 0.813
Epoch 1, Batch 2/21, Loss: 0.721
Epoch 1, Batch 3/21, Loss: 0.609
Epoch 1, Batch 4/21, Loss: 0.585
Epoch 1, Batch 5/21, Loss: 0.544
Epoch 1, Batch 6/21, Loss: 0.575
Epoch 1, Batch 7/21, Loss: 0.514
Epoch 1, Batch 8/21, Loss: 0.587
Epoch 1, Batch 9/21, Loss: 0.543
Epoch 1, Batch 10/21, Loss: 0.545
Epoch 1, Batch 11/21, Loss: 0.546
Epoch 1, Batch 12/21, Loss: 0.498
Epoch 1, Batch 13/21, Loss: 0.421
Epoch 1, Batch 14/21, Loss: 0.500
Epoch 1, Batch 15/21, Loss: 0.556
Epoch 1, Batch 16/21, Loss: 0.522
Epoch 1, Batch 17/21, Loss: 0.546
Epoch 1, Batch 18/21, Loss: 0.442
Epoch 1, Batch 19/21, Loss: 0.528
Epoch 1, Batch 20/21, Loss: 0.410
Epoch 1, Batch 21/21, Loss: 0.502
Epoch 1 finished!
Epoch 2, Batch 1/21, Loss: 0.543
Epoch 2, Batch 2/21, Loss: 0.426
Epoch 2, Batch 3/21, Loss: 0.528
Epoch 2, Batch 4/21, Loss: 0.516
Epoch 2, Batch 5/21, Loss: 0.529
Epoch 2, Batch 6/21, Loss: 0.471
Epoch 2, Batch 7/21, Loss: 0.456
Epoch 2, Batch 8/21, Loss: 0.459
Epoch 2, Batc

In [66]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# Constants
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 0.0001
EPOCHS = 20
BS = 32

# Define Dataset
class MyDataset(Dataset):
    def __init__(self, data_file, is_train=True):
        df = pd.read_csv(data_file)
        self.is_train = is_train
        self.inputs = df.iloc[:, :-1].values if is_train else df.values
        self.labels = df.iloc[:, -1].values if is_train else None
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        inp = self.inputs[idx].astype(np.float32)
        if self.is_train:
            lab = self.labels[idx].astype(np.float32)
            return torch.tensor(inp), torch.tensor(lab).unsqueeze(0)
        else:
            return torch.tensor(inp)

# Load Data
tr_ds = MyDataset('train.csv')
te_ds = MyDataset('test.csv', is_train=False)

tr_dl = DataLoader(tr_ds, batch_size=BS, shuffle=True)
te_dl = DataLoader(te_ds, batch_size=BS, shuffle=False)

# Model
'''
model = nn.Sequential(
    nn.Linear(16, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#77
'''
'''
model = nn.Sequential(
    nn.Linear(16, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.2),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#78
'''
'''
model = nn.Sequential(
    nn.Linear(16, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#43
'''
'''
model = nn.Sequential(
    nn.Linear(16, 512),  nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(256, 128),  nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#57
'''
model = nn.Sequential(
    nn.Linear(16, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Train & Evaluate
def train_and_evaluate(model, tr_dl, te_dl, criterion, optimizer, epochs, device):
    for epoch in range(epochs):
        model.train()
        for i, (inputs, labels) in enumerate(tr_dl):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(); outputs = model(inputs); loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()
            print(f'Epoch {epoch+1}, Batch {i+1}/{len(tr_dl)}, Loss: {loss.item():.3f}')
        print(f'Epoch {epoch+1} finished!')

    model.eval()
    preds = []
    with torch.no_grad():
        for inputs in te_dl:
            inputs = inputs.to(device); outputs = model(inputs)
            preds.extend((outputs > 0.5).int().flatten().cpu().numpy())
    return preds

# Run
preds = train_and_evaluate(model, tr_dl, te_dl, criterion, optimizer, EPOCHS, DEV)
df_answers = pd.DataFrame(preds, columns=['predictions']); df_answers.to_csv('answers.csv', index=False)

# Calculate accuracy
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in tr_dl:
        inputs, labels = inputs.to(DEV), labels.to(DEV)
        outputs = model(inputs)
        predicted = (outputs > 0.5).int()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the training data: {accuracy:.2f}%')

Epoch 1, Batch 1/41, Loss: 0.681
Epoch 1, Batch 2/41, Loss: 0.748
Epoch 1, Batch 3/41, Loss: 0.738
Epoch 1, Batch 4/41, Loss: 0.680
Epoch 1, Batch 5/41, Loss: 0.662
Epoch 1, Batch 6/41, Loss: 0.681
Epoch 1, Batch 7/41, Loss: 0.682
Epoch 1, Batch 8/41, Loss: 0.630
Epoch 1, Batch 9/41, Loss: 0.638
Epoch 1, Batch 10/41, Loss: 0.664
Epoch 1, Batch 11/41, Loss: 0.638
Epoch 1, Batch 12/41, Loss: 0.716
Epoch 1, Batch 13/41, Loss: 0.616
Epoch 1, Batch 14/41, Loss: 0.687
Epoch 1, Batch 15/41, Loss: 0.647
Epoch 1, Batch 16/41, Loss: 0.714
Epoch 1, Batch 17/41, Loss: 0.649
Epoch 1, Batch 18/41, Loss: 0.660
Epoch 1, Batch 19/41, Loss: 0.597
Epoch 1, Batch 20/41, Loss: 0.593
Epoch 1, Batch 21/41, Loss: 0.581
Epoch 1, Batch 22/41, Loss: 0.653
Epoch 1, Batch 23/41, Loss: 0.647
Epoch 1, Batch 24/41, Loss: 0.556
Epoch 1, Batch 25/41, Loss: 0.585
Epoch 1, Batch 26/41, Loss: 0.591
Epoch 1, Batch 27/41, Loss: 0.595
Epoch 1, Batch 28/41, Loss: 0.629
Epoch 1, Batch 29/41, Loss: 0.632
Epoch 1, Batch 30/41, L

In [68]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd

# Constants
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 0.0001
EPOCHS = 20
BS = 32
SEED = 42  # Random seed for reproducibility

# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Define Dataset
class MyDataset(Dataset):
    def __init__(self, data_file, is_train=True, inputs=None, labels=None):
        if inputs is None and labels is None:
            df = pd.read_csv(data_file)
            self.inputs = df.iloc[:, :-1].values if is_train else df.values
            self.labels = df.iloc[:, -1].values if is_train else None
        else:
            self.inputs = inputs
            self.labels = labels
        self.is_train = is_train
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        inp = self.inputs[idx].astype(np.float32)
        if self.is_train:
            lab = self.labels[idx].astype(np.float32)
            return torch.tensor(inp), torch.tensor(lab).unsqueeze(0)
        else:
            return torch.tensor(inp)

# Load Data
df = pd.read_csv('train.csv')
inputs = df.iloc[:, :-1].values
labels = df.iloc[:, -1].values

# Split data into training and validation sets
dataset = MyDataset('train.csv')
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

# Create DataLoaders
tr_dl = DataLoader(train_dataset, batch_size=BS, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=BS, shuffle=False)
te_ds = MyDataset('test.csv', is_train=False)
te_dl = DataLoader(te_ds, batch_size=BS, shuffle=False)

# Model

model = nn.Sequential(
    nn.Linear(16, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#77

'''
model = nn.Sequential(
    nn.Linear(16, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.2),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#78
'''
'''
model = nn.Sequential(
    nn.Linear(16, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#43
'''
'''
model = nn.Sequential(
    nn.Linear(16, 512),  nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(256, 128),  nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#57
'''
'''
model = nn.Sequential(
    nn.Linear(16, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
'''
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Train & Evaluate
def train_and_evaluate(model, tr_dl, val_dl, te_dl, criterion, optimizer, epochs, device):
    for epoch in range(epochs):
        model.train()
        for i, (inputs, labels) in enumerate(tr_dl):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(); outputs = model(inputs); loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()
            print(f'Epoch {epoch+1}, Batch {i+1}/{len(tr_dl)}, Loss: {loss.item():.3f}')
        print(f'Epoch {epoch+1} finished!')

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                predicted = (outputs > 0.5).int()
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(val_dl)
        val_accuracy = 100 * correct / total
        print(f'Epoch {epoch+1}, Validation Loss: {val_loss:.3f}, Validation Accuracy: {val_accuracy:.2f}%')

    model.eval()
    preds = []
    with torch.no_grad():
        for inputs in te_dl:
            inputs = inputs.to(device); outputs = model(inputs)
            preds.extend((outputs > 0.5).int().flatten().cpu().numpy())
    return preds

# Run
preds = train_and_evaluate(model, tr_dl, val_dl, te_dl, criterion, optimizer, EPOCHS, DEV)
df_answers = pd.DataFrame(preds, columns=['predictions']); df_answers.to_csv('answers.csv', index=False)

# Calculate accuracy on the training data
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in tr_dl:
        inputs, labels = inputs.to(DEV), labels.to(DEV)
        outputs = model(inputs)
        predicted = (outputs > 0.5).int()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the training data: {accuracy:.2f}%')

Epoch 1, Batch 1/33, Loss: 0.721
Epoch 1, Batch 2/33, Loss: 0.775
Epoch 1, Batch 3/33, Loss: 0.715
Epoch 1, Batch 4/33, Loss: 0.803
Epoch 1, Batch 5/33, Loss: 0.793
Epoch 1, Batch 6/33, Loss: 0.792
Epoch 1, Batch 7/33, Loss: 0.735
Epoch 1, Batch 8/33, Loss: 0.784
Epoch 1, Batch 9/33, Loss: 0.662
Epoch 1, Batch 10/33, Loss: 0.720
Epoch 1, Batch 11/33, Loss: 0.722
Epoch 1, Batch 12/33, Loss: 0.695
Epoch 1, Batch 13/33, Loss: 0.704
Epoch 1, Batch 14/33, Loss: 0.742
Epoch 1, Batch 15/33, Loss: 0.710
Epoch 1, Batch 16/33, Loss: 0.747
Epoch 1, Batch 17/33, Loss: 0.740
Epoch 1, Batch 18/33, Loss: 0.678
Epoch 1, Batch 19/33, Loss: 0.665
Epoch 1, Batch 20/33, Loss: 0.724
Epoch 1, Batch 21/33, Loss: 0.684
Epoch 1, Batch 22/33, Loss: 0.695
Epoch 1, Batch 23/33, Loss: 0.703
Epoch 1, Batch 24/33, Loss: 0.675
Epoch 1, Batch 25/33, Loss: 0.690
Epoch 1, Batch 26/33, Loss: 0.647
Epoch 1, Batch 27/33, Loss: 0.613
Epoch 1, Batch 28/33, Loss: 0.707
Epoch 1, Batch 29/33, Loss: 0.568
Epoch 1, Batch 30/33, L

In [90]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler  # Import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau # Import Learning Rate Scheduler
import copy

# Constants
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 0.0001
EPOCHS = 500
BS = 32
SEED = 42  # Random seed for reproducibility
EARLY_STOPPING_PATIENCE = 5 # Number of epochs to wait for validation loss improvement
LR_SCHEDULER_PATIENCE = 3
LR_SCHEDULER_FACTOR = 0.5

# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Define Dataset
class MyDataset(Dataset):
    def __init__(self, data_file, is_train=True, inputs=None, labels=None, scaler=None): # Add scaler
        if inputs is None and labels is None:
            df = pd.read_csv(data_file)
            self.inputs = df.iloc[:, :-1].values
            self.labels = df.iloc[:, -1].values
        else:
            self.inputs = inputs
            self.labels = labels
        self.is_train = is_train
        self.scaler = scaler # Store the scaler

        if self.scaler is not None and self.is_train:
             self.inputs = self.scaler.fit_transform(self.inputs)  # Fit and transform ONLY on training data
        elif self.scaler is not None:
            self.inputs = self.scaler.transform(self.inputs) # Transform on validation/test data


    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        inp = self.inputs[idx].astype(np.float32)
        if self.is_train:
            lab = self.labels[idx].astype(np.float32)
            return torch.tensor(inp), torch.tensor(lab).unsqueeze(0)
        else:
            return torch.tensor(inp)

# Load Data
df = pd.read_csv('train.csv')
inputs = df.iloc[:, :-1].values
labels = df.iloc[:, -1].values

# Split data into training and validation sets

# Create a StandardScaler instance
scaler = StandardScaler() # Create the scaler here!


dataset = MyDataset('train.csv', scaler=scaler)  # Pass the scaler to the dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

# Create DataLoaders
tr_dl = DataLoader(train_dataset, batch_size=BS, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=BS, shuffle=False)

# Create the test dataset and dataloader
test_df = pd.read_csv('test.csv') #load test data
test_inputs = test_df.values
test_dataset = MyDataset('test.csv', is_train=False, inputs=test_inputs, labels=None, scaler=scaler) # Pass scaler and data
te_dl = DataLoader(test_dataset, batch_size=BS, shuffle=False)



# Model
'''
model = nn.Sequential(
    nn.Linear(16, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#77
'''

model = nn.Sequential(
    nn.Linear(16, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.2),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#78

'''
model = nn.Sequential(
    nn.Linear(16, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#43
'''
'''
model = nn.Sequential(
    nn.Linear(16, 512),  nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(p=0.6),
    nn.Linear(256, 128),  nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
#57
'''
'''
model = nn.Sequential(
    nn.Linear(16, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=0.4),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=0.2),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEV)
'''

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=LR_SCHEDULER_FACTOR, patience=LR_SCHEDULER_PATIENCE, verbose=True)

# Train & Evaluate
def train_and_evaluate(model, tr_dl, val_dl, te_dl, criterion, optimizer, scheduler, epochs, device):
    best_val_loss = float('inf')  # Initialize best validation loss
    epochs_no_improve = 0 # Counter for early stopping
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        for i, (inputs, labels) in enumerate(tr_dl):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(); outputs = model(inputs); loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()
            print(f'Epoch {epoch+1}, Batch {i+1}/{len(tr_dl)}, Loss: {loss.item():.3f}')

        # Validation loop:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0) # Accumulate validation loss

        val_loss /= len(val_dl.dataset) # Get average validation loss

        print(f'Epoch {epoch+1} finished! Validation Loss: {val_loss:.3f}')

         # --- Learning Rate Scheduler Step ---
        scheduler.step(val_loss)

        # Early stopping check:
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            if epochs_no_improve == EARLY_STOPPING_PATIENCE:
                print(f'Early stopping triggered after {EARLY_STOPPING_PATIENCE} epochs!')
                break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model state based on validation loss.")
    else:
        print("No improvement in validation loss, using the last model state.")

    model.eval()
    preds = []
    with torch.no_grad():
        for inputs in te_dl:
            inputs = inputs.to(device); outputs = model(inputs)
            preds.extend((outputs > 0.5).int().flatten().cpu().numpy())
    return preds

# Run
preds = train_and_evaluate(model, tr_dl, val_dl, te_dl, criterion, optimizer, scheduler, EPOCHS, DEV)
df_answers = pd.DataFrame(preds, columns=['predictions']); df_answers.to_csv('answers.csv', index=False,header=False)

# Calculate accuracy on the training data
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in tr_dl:
        inputs, labels = inputs.to(DEV), labels.to(DEV)
        outputs = model(inputs)
        predicted = (outputs > 0.5).int()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the training data: {accuracy:.2f}%')

Epoch 1, Batch 1/33, Loss: 0.673
Epoch 1, Batch 2/33, Loss: 0.752
Epoch 1, Batch 3/33, Loss: 0.697
Epoch 1, Batch 4/33, Loss: 0.717
Epoch 1, Batch 5/33, Loss: 0.722
Epoch 1, Batch 6/33, Loss: 0.713
Epoch 1, Batch 7/33, Loss: 0.687
Epoch 1, Batch 8/33, Loss: 0.701
Epoch 1, Batch 9/33, Loss: 0.640
Epoch 1, Batch 10/33, Loss: 0.653
Epoch 1, Batch 11/33, Loss: 0.620
Epoch 1, Batch 12/33, Loss: 0.681
Epoch 1, Batch 13/33, Loss: 0.692
Epoch 1, Batch 14/33, Loss: 0.663
Epoch 1, Batch 15/33, Loss: 0.666
Epoch 1, Batch 16/33, Loss: 0.629
Epoch 1, Batch 17/33, Loss: 0.628
Epoch 1, Batch 18/33, Loss: 0.622
Epoch 1, Batch 19/33, Loss: 0.646
Epoch 1, Batch 20/33, Loss: 0.568
Epoch 1, Batch 21/33, Loss: 0.605
Epoch 1, Batch 22/33, Loss: 0.661
Epoch 1, Batch 23/33, Loss: 0.532
Epoch 1, Batch 24/33, Loss: 0.609
Epoch 1, Batch 25/33, Loss: 0.604
Epoch 1, Batch 26/33, Loss: 0.629
Epoch 1, Batch 27/33, Loss: 0.624
Epoch 1, Batch 28/33, Loss: 0.590
Epoch 1, Batch 29/33, Loss: 0.620
Epoch 1, Batch 30/33, L

In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest

# Load the training data
try:
    train_df = pd.read_csv('train.csv')
    print("Training data loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv not found.")
    exit()

# --- 1. Data Exploration and Visualization ---

# --- Summary Statistics ---
print("\n--- Summary Statistics ---")
print(train_df.describe())

# --- Check for Missing Values ---
print("\n--- Missing Values ---")
print(train_df.isnull().sum())  # Show missing values per column

# --- Visualize Feature Distributions ---
print("\n--- Visualizing Feature Distributions ---")
train_df.hist(figsize=(20, 15))
plt.suptitle("Histograms of Features", fontsize=16)
plt.show()

# --- Target Variable Distribution ---
print("\n--- Target Variable Distribution ---")
sns.countplot(x='target', data=train_df)
plt.title("Distribution of Target Variable")
plt.show()

# --- Distributions of Features by Target Class ---
print("\n--- Distributions of Features by Target Class ---")
for col in train_df.columns[:-1]:  # Exclude target column
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='target', y=col, data=train_df)
    plt.title(f'{col} Distribution by Target Class')
    plt.show()

# --- Pairwise Relationships (Limited to a subset of features for clarity) ---
print("\n--- Pairwise Relationships (Limited to a subset of features) ---")
subset_cols = train_df.columns[:5].tolist() + ['target']  # First 5 features + target
sns.pairplot(train_df[subset_cols], hue='target')
plt.suptitle("Pairplot of a Subset of Features", y=1.02)
plt.show()

# --- 2. Outlier Detection ---

# --- Isolation Forest ---
print("\n--- Outlier Detection using Isolation Forest ---")
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
outlier_predictions = iso_forest.fit_predict(train_df.drop('target', axis=1))

# Add outlier predictions to the dataframe
train_df['outlier'] = outlier_predictions

# Examine the number of outliers
print("\nNumber of outliers:", train_df[train_df['outlier'] == -1].shape[0])

# --- Visualize Outliers ---
print("\n--- Visualizing Outliers ---")
# Scatter plot of two features, colored by outlier status
plt.figure(figsize=(10, 8))
sns.scatterplot(x=train_df.columns[0], y=train_df.columns[1], hue='outlier', data=train_df)
plt.title("Outlier Visualization using Two Features")
plt.show()


# --- 3. Handling Outliers ---
print("\n--- Handling Outliers (Clipping) ---")

def clip_outliers(df, columns, threshold=3):  # Using clipping method
    for col in columns:
        mean = df[col].mean()
        std = df[col].std()
        upper_bound = mean + threshold * std
        lower_bound = mean - threshold * std
        df[col] = np.clip(df[col], lower_bound, upper_bound)
    return df

# Clip outliers
numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()
numeric_cols.remove('target') # Remove target before clipping

train_df_clipped = clip_outliers(train_df.copy(), numeric_cols) # Avoid modifying original

# Verify the effect of clipping using box plots
print("\n--- Comparing Original and Clipped Data (Box Plots) ---")
for col in train_df.columns[:-2]:  # Show box plots before and after clipping
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sns.boxplot(ax=axes[0], y=train_df[col])
    axes[0].set_title(f'Original {col} Distribution')

    sns.boxplot(ax=axes[1], y=train_df_clipped[col])
    axes[1].set_title(f'Clipped {col} Distribution')

    plt.show()


# --- 4. Imputation Strategies (if missing values exist) ---
print("\n--- Imputation Strategies (No Missing Values) ---")
print("No missing values detected in the dataset. Skipping imputation.")

Output hidden; open in https://colab.research.google.com to view.

In [112]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split  # Import train_test_split
from sklearn.preprocessing import StandardScaler  # Import StandardScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau # Import Learning Rate Scheduler
import copy

# Constants
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR = 0.0005  # Slightly increased learning rate
EPOCHS = 500
BS = 32  # Increased batch size
SEED = 42  # Random seed for reproducibility
EARLY_STOPPING_PATIENCE = 10 # Increased patience
LR_SCHEDULER_PATIENCE = 5 # Reduced patience to be more responsive
LR_SCHEDULER_FACTOR = 0.3 # Aggressive reduction factor

# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Define Dataset
class MyDataset(Dataset):
    def __init__(self, inputs, labels):  # Simplified init
        self.inputs = inputs.astype(np.float32)
        self.labels = labels.astype(np.float32)
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx]), torch.tensor(self.labels[idx]).unsqueeze(0)


# Load Data
df = pd.read_csv('train.csv')
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# --- Train/Validation Split with Stratification ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=SEED, stratify=y
)

# --- Scaling ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # Fit and transform on training data
X_val = scaler.transform(X_val) # Transform validation data

# Create datasets and dataloaders
train_dataset = MyDataset(X_train, y_train)
val_dataset = MyDataset(X_val, y_val)
tr_dl = DataLoader(train_dataset, batch_size=BS, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=BS, shuffle=False)

# Load and scale test data
test_df = pd.read_csv('test.csv')
test_inputs = test_df.values
test_inputs = scaler.transform(test_inputs) # Scale test data
test_dataset = MyDataset(test_inputs, np.zeros(len(test_inputs)))  # Dummy labels, required for dataloader

te_dl = DataLoader(test_dataset, batch_size=BS, shuffle=False)


'''
class SimpleNN(nn.Module):  # Define as a class for more flexibility
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.seq = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),  # Less dropout in the final layer
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.seq(x)
# 88
'''

class SimpleNN(nn.Module):  # Define as a class for more flexibility
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.seq = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),  # Less dropout in the final layer
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),  # Less dropout in the final layer
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.seq(x)
# 88.7

'''
class SimpleNN(nn.Module):  # Define as a class for more flexibility
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.seq = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),  # Less dropout in the final layer
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),  # Less dropout in the final layer
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.seq(x)
## 88.78
'''
model = SimpleNN(X_train.shape[1]).to(DEV)  # Instantiate the model

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LR,weight_decay=1e-6)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=LR_SCHEDULER_FACTOR, patience=LR_SCHEDULER_PATIENCE, verbose=True)

# Train & Evaluate
def train_and_evaluate(model, tr_dl, val_dl, te_dl, criterion, optimizer, scheduler, epochs, device):
    best_val_loss = float('inf')  # Initialize best validation loss
    epochs_no_improve = 0 # Counter for early stopping
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for i, (inputs, labels) in enumerate(tr_dl):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(); outputs = model(inputs); loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()
            train_loss += loss.item() * inputs.size(0)
        train_loss /= len(tr_dl.dataset)

        # Validation loop:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_dl:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0) # Accumulate validation loss

        val_loss /= len(val_dl.dataset) # Get average validation loss

        print(f'Epoch {epoch+1} finished! Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')

         # --- Learning Rate Scheduler Step ---
        scheduler.step(val_loss)

        # Early stopping check:
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            if epochs_no_improve == EARLY_STOPPING_PATIENCE:
                print(f'Early stopping triggered after {EARLY_STOPPING_PATIENCE} epochs!')
                break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model state based on validation loss.")
    else:
        print("No improvement in validation loss, using the last model state.")

    model.eval()
    preds = []
    with torch.no_grad():
        for inputs, _ in te_dl:  # No labels for test data
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds.extend((outputs > 0.5).int().flatten().cpu().numpy())
    return preds

# Run
preds = train_and_evaluate(model, tr_dl, val_dl, te_dl, criterion, optimizer, scheduler, EPOCHS, DEV)
df_answers = pd.DataFrame(preds, columns=['predictions']); df_answers.to_csv('answers.csv', index=False, header=False)

# Calculate accuracy on the training data - not very useful, just for reference
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in tr_dl:
        inputs, labels = inputs.to(DEV), labels.to(DEV)
        outputs = model(inputs)
        predicted = (outputs > 0.5).int()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the training data: {accuracy:.2f}%')

Epoch 1 finished! Train Loss: 0.5065, Validation Loss: 0.3854
Epoch 2 finished! Train Loss: 0.3873, Validation Loss: 0.3355
Epoch 3 finished! Train Loss: 0.3810, Validation Loss: 0.3081
Epoch 4 finished! Train Loss: 0.3584, Validation Loss: 0.3003
Epoch 5 finished! Train Loss: 0.3572, Validation Loss: 0.3020
Epoch 6 finished! Train Loss: 0.3583, Validation Loss: 0.2899
Epoch 7 finished! Train Loss: 0.3369, Validation Loss: 0.2772
Epoch 8 finished! Train Loss: 0.3338, Validation Loss: 0.2724
Epoch 9 finished! Train Loss: 0.3596, Validation Loss: 0.2779
Epoch 10 finished! Train Loss: 0.3507, Validation Loss: 0.2636
Epoch 11 finished! Train Loss: 0.3376, Validation Loss: 0.2622
Epoch 12 finished! Train Loss: 0.3457, Validation Loss: 0.2645
Epoch 13 finished! Train Loss: 0.3401, Validation Loss: 0.2686
Epoch 14 finished! Train Loss: 0.3394, Validation Loss: 0.2653
Epoch 15 finished! Train Loss: 0.3226, Validation Loss: 0.2612
Epoch 16 finished! Train Loss: 0.3715, Validation Loss: 0.2789
E